#Objective: Analyze sales data using SQL with filtering, aggregation, and business queries.

In [ ]:
pip install pandas sqlalchemy pymysql


1) Load dataset into a SQL database.

In [ ]:
import pandas as pd
df = pd.read_csv("Sample - Superstore.csv", encoding="latin1")
print(df.shape)
df.head()

Each row represents a product line item within an order. Multiple rows can belong to the same Order ID because one order may contain multiple products.

2) Explore table (schema, sample data).

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df.dtypes

The dataset does not explicitly define a primary key.
However, the Row ID column contains a unique value for every record and can be considered a candidate primary key.
The Order ID column cannot be used as a primary key because a single order may contain multiple products, resulting in multiple rows with the same Order ID.

In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])
df.dtypes

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

In [ ]:
df.isnull().sum()

In [ ]:
print("Duplicate Rows:", df.duplicated().sum())

In [ ]:
df[df['Region'] == 'West']

3. Apply WHERE filters (region, category, date, sales).

#Equivalent sql
SELECT *
FROM superstore_sales
WHERE Region = 'West';

In [ ]:
df[df['Category'] == 'Technology']

In [ ]:
df[df['Sales'] > 1000]

In [ ]:
df[df['Profit'] < 0]

In [ ]:
df[(df['Sales'] >= 500) & (df['Sales'] <= 1000)]

In [ ]:
import sqlite3

conn = sqlite3.connect('sales.db')
df.to_sql('superstore_sales', conn, if_exists='replace', index=False)

In [ ]:
query = """
SELECT *
FROM superstore_sales
WHERE Region = 'West'
"""

import pandas as pd
pd.read_sql(query, conn)

In [ ]:
pd.read_sql("""
SELECT *
FROM superstore_sales
WHERE Category = 'Technology'
LIMIT 5
""", conn)

In [ ]:
pd.read_sql("""
SELECT *
FROM superstore_sales
WHERE Sales > 1000

""", conn)

In [ ]:
pd.read_sql("""
SELECT *
FROM superstore_sales
WHERE Profit < 0
LIMIT 5
""", conn)

In [ ]:
pd.read_sql("""
SELECT *
FROM superstore_sales
WHERE Region = 'West'
AND Category = 'Technology'
LIMIT 5
""", conn)

#4) Group By Aggregations


Query 1: Total Sales by Region

In [ ]:
pd.read_sql("""
SELECT
    Region,
    SUM(Sales) AS Total_Sales
FROM superstore_sales
GROUP BY Region;
""", conn)

Total Sales by Category

In [ ]:
pd.read_sql("""
SELECT
    Category,
    SUM(Sales) AS Total_Sales
FROM superstore_sales
GROUP BY Category
ORDER BY Total_Sales DESC;
""", conn)

Total Quantity Sold by Category

In [ ]:
pd.read_sql("""
SELECT
    Category,
    SUM(Quantity) AS Total_Quantity
FROM superstore_sales
GROUP BY Category
ORDER BY Total_Quantity DESC;
""", conn)

Average Sales by Category

In [ ]:
pd.read_sql("""
SELECT
    Category,
    AVG(Sales) AS Avg_Sales
FROM superstore_sales
GROUP BY Category
ORDER BY Avg_Sales DESC;
""", conn)

5. Sort and limit results (top products, top categories).

In [ ]:
#Top 10 Products by Sales
pd.read_sql("""
SELECT
    `Product Name`,
    SUM(Sales) AS Total_Sales
FROM superstore_sales
GROUP BY `Product Name`
ORDER BY Total_Sales DESC
LIMIT 10;
""", conn)

In [ ]:
#Top 10 Customers by Sales
pd.read_sql("""
SELECT
    `Customer Name`,
    SUM(Sales) AS Total_Sales
FROM superstore_sales
GROUP BY `Customer Name`
ORDER BY Total_Sales DESC
LIMIT 10;
""", conn)

In [ ]:
#Bottom 5 Products by Profit
pd.read_sql("""
SELECT
    `Product Name`,
    SUM(Profit) AS Total_Profit
FROM superstore_sales
GROUP BY `Product Name`
ORDER BY Total_Profit ASC
LIMIT 5;
""", conn)

#6.Solve use cases (monthly trends, top customers, duplicates).

In [ ]:
# Highest Sales Days
pd.read_sql("""
SELECT
    `Order Date`,
    SUM(Sales) AS Total_Sales
FROM superstore_sales
GROUP BY `Order Date`
ORDER BY Total_Sales DESC
LIMIT 10;
""", conn)

In [ ]:
# Most Profitable Days
pd.read_sql("""
SELECT
    `Order Date`,
    SUM(Profit) AS Total_Profit
FROM superstore_sales
GROUP BY `Order Date`
ORDER BY Total_Profit DESC
LIMIT 10;
""", conn)

In [ ]:
# Top 10 Customers by Sales
pd.read_sql("""
SELECT
    `Customer Name`,
    SUM(Sales) AS Total_Sales
FROM superstore_sales
GROUP BY `Customer Name`
ORDER BY Total_Sales DESC
LIMIT 10;
""", conn)

In [ ]:
# Customers with Most Orders
pd.read_sql("""
SELECT
    `Customer Name`,
    COUNT(`Order ID`) AS Total_Orders
FROM superstore_sales
GROUP BY `Customer Name`
ORDER BY Total_Orders DESC
LIMIT 10;
""", conn)

In [ ]:
# Order IDs Appearing More Than Once
pd.read_sql("""
SELECT
    `Order ID`,
    COUNT(*) AS Count
FROM superstore_sales
GROUP BY `Order ID`
HAVING COUNT(*) > 1
ORDER BY Count DESC;
""", conn)

7. Validate results (row counts, data quality).

In [ ]:
pd.read_sql("""
SELECT COUNT(*) AS Total_Rows
FROM superstore_sales;
""", conn)

In [ ]:
pd.read_sql("""
SELECT COUNT(DISTINCT `Customer ID`) AS Total_Customers
FROM superstore_sales;
""", conn)

In [ ]:
# Highest and Lowest Sales
pd.read_sql("""
SELECT
    MAX(Sales) AS Highest_Sale,
    MIN(Sales) AS Lowest_Sale
FROM superstore_sales;
""", conn)

In [ ]:
# Highest and Lowest Profit
pd.read_sql("""
SELECT
    MAX(Profit) AS Highest_Profit,
    MIN(Profit) AS Lowest_Profit
FROM superstore_sales;
""", conn)

In [ ]:
# Discount vs Profit Analysis
pd.read_sql("""
SELECT
    Category,
    AVG(Discount) AS Avg_Discount,
   AVG(Profit) AS Avg_Profit
FROM superstore_sales
GROUP BY Category
ORDER BY Avg_Profit DESC;
""", conn)

# Key Findings and Insights

1. The dataset contains 9,994 sales transactions across multiple regions, categories, and customers.

2. No missing values or duplicate rows were identified, indicating good data quality.

3. The West region generated the highest sales revenue among all regions.

4. Technology emerged as one of the strongest-performing categories in terms of sales.

5. A small group of customers contributed significantly to total revenue, highlighting the importance of customer retention.

6. Certain products generated losses despite recording high sales, indicating the impact of discounts on profitability.

7. Multiple occurrences of the same Order ID were observed. This is expected because one order can contain multiple products.
